# Spatial Detection — Foreign-Target Comparison (final)

Compares our detectors on **foreign** (out-of-scene) targets across three scenes.

**Detectors** (toggle in any cell via `active_detectors`):
`DSM` · `NeighborMLP` · `NeighborMLP-CFAR` · `AMF` · `GMM-Levin`
*(More baselines can be added later — see the registry in `run_compare.DET_ORDER`.)*

**Scenes:** 0 (manual box 0), 2 (manual box 2), 4 (random box, seed 42).
Each scene cell runs **foreign targets only** — the target class is one that is
**absent** from the scene (listed in the cell comment). Tune `foreign_class`,
`amplitude`, `n_budget`, `k`, and `active_detectors` per cell.

In [ ]:
# ── Cell 1: Clone repo + install deps ─────────────────────────────────
import os, sys

REPO_URL      = 'https://github.com/michaelpiro/final-paper-experiment.git'
LOCAL_PROJECT = '/content/proj'
BRANCH        = 'iid-unified-experiment'

if os.path.exists(os.path.join(LOCAL_PROJECT, '.git')):
    !git -C {LOCAL_PROJECT} fetch --all -q
    !git -C {LOCAL_PROJECT} checkout {BRANCH} -q
    !git -C {LOCAL_PROJECT} pull origin {BRANCH} -q
else:
    !git clone -b {BRANCH} --depth 1 {REPO_URL} {LOCAL_PROJECT}

!pip install -q scikit-learn scipy tqdm matplotlib pyyaml pandas

for p in [LOCAL_PROJECT, os.path.join(LOCAL_PROJECT, 'experiments', 'spatial')]:
    if p not in sys.path:
        sys.path.insert(0, p)
os.chdir(LOCAL_PROJECT)
print('Ready. CWD:', os.getcwd())
!git -C {LOCAL_PROJECT} log --oneline -1

In [ ]:
# ── Cell 2: GPU check ─────────────────────────────────────────────────
import torch
DEVICE = 'cuda' if torch.cuda.is_available() else 'cpu'
print('Device:', DEVICE, '|', torch.cuda.get_device_name(0) if DEVICE=='cuda' else 'enable GPU runtime')

In [ ]:
# ── Cell 3: Mount Drive + link pavia-u.mat ────────────────────────────
import os
RESULTS = '/content/results'
try:
    from google.colab import drive
    drive.mount('/content/drive')
    RESULTS = '/content/drive/MyDrive/final_paper/spatial_results'
    os.makedirs('/content/proj/data', exist_ok=True)
    DST = '/content/proj/data/pavia-u.mat'
    if not os.path.exists(DST):
        for src in ['/content/drive/MyDrive/final_paper/pavia-u.mat',
                    '/content/drive/MyDrive/final_paper/real_datasets/pavia-u.mat']:
            if os.path.exists(src):
                os.symlink(src, DST); print('Linked dataset from', src); break
        else:
            print('WARNING: pavia-u.mat not found on Drive.')
    else:
        print('Dataset already linked.')
except Exception as e:
    print('Drive not mounted:', repr(e))
os.makedirs(RESULTS, exist_ok=True)
assert os.path.exists('/content/proj/data/pavia-u.mat'), 'pavia-u.mat missing!'
print('RESULTS dir:', RESULTS)

In [ ]:
# ── Cell 4: Helpers (reload + run + show) ─────────────────────────────
import importlib, sys, os
import run_compare as rc
importlib.reload(rc)

DATASET = '/content/proj/data/pavia-u.mat'
MANUAL  = '/content/proj/experiments/spatial/manual_boxes.json'

def run_and_show(ov, show=True):
    """Reload code, run one scene, show the foreign-target results."""
    importlib.reload(rc)
    ov = dict(ov)
    ov.setdefault('dataset', DATASET)
    ov.setdefault('manual_boxes_path', MANUAL)
    ov.setdefault('results_dir', RESULTS)
    rd = rc.run_from_cfg(ov)
    if show:
        sub = 'foreign' if ov.get('run_foreign', True) else None
        rc.show_plots_from_dir(rd, sub=sub)
    return rd

print('Helpers ready. DET_ORDER =', rc.DET_ORDER)

In [ ]:
# ── Cell 5: BASE config (edit defaults here; override per scene below) ─
BASE = dict(
    # ── scene / protocol ──
    norm_mode         = 'none',
    sig_dom_weight    = 0.8,
    sig_mean_weight   = 0.2,
    random_scenario_seeds = [42, 123, 456, 789],
    min_pixels        = 3000,

    # ── run only FOREIGN targets ──
    run_inpatch       = False,
    run_foreign       = True,

    # ── which detectors to run/show (omit to run all of DET_ORDER) ──
    active_detectors  = ['DSM', 'NeighborMLP', 'NeighborMLP-CFAR', 'AMF', 'GMM-Levin'],

    # ── spatial window (same k for ALL spatial detectors) ──
    k                 = 5,

    # ── target planting ──
    amplitude         = 0.15,
    target_fraction   = 0.10,
    edge_guard        = 3,
    n_budget          = None,         # None = full train box; int = side-crop

    # ── DSM (global) ──
    dsm_hidden        = [128],
    dsm_epochs        = 1000,
    dsm_lr            = 5e-4,

    # ── NeighborMLP ──
    nmlp_d_lat        = 16,
    nmlp_K            = 7,
    nmlp_enc_hidden   = [64, 32],
    nmlp_score_hidden = [128],
    nmlp_epochs       = 1000,
    nmlp_lr           = 3e-4,
    nmlp_batch        = 512,

    # ── GMM-Levin ──
    gmm_K             = 9,
    gmm_steps         = 50,

    # ── baselines ──
    local_scm_loading = 1e-12,
    baseline_eig_floor= 1e-12,

    # ── shared ──
    activation        = 'relu',
    dsm_sigma_rho     = 0.05,
    whiten_mode       = 'zca',
    whiten_eig_floor  = 1e-5,
    batch_size        = 256,
    weight_decay      = 1e-5,
    pfa_target        = 0.05,
    seed              = 42,
)
print('BASE config ready.  active_detectors =', BASE['active_detectors'])

In [ ]:
# ── Cell 6: Scene overview — full false-color image + GT label map ────
import numpy as np, json, matplotlib.pyplot as plt
from matplotlib.colors import to_rgb
import matplotlib.patches as mpatches
from final_paper_experiments.data_utils import load_and_normalize
from final_paper_experiments.evaluation import generate_random_boxes

CLS_NAMES  = {0:'unlabeled',1:'asphalt',2:'meadows',3:'gravel',4:'trees',
              5:'metal_sheets',6:'bare_soil',7:'bitumen',8:'bricks',9:'shadows'}
CLS_HEX    = {0:'#000000',1:'#808080',2:'#00cc44',3:'#d2691e',4:'#006400',
              5:'#add8e6',6:'#a52a2a',7:'#9400d3',8:'#ff4500',9:'#00008b'}

data, gt = load_and_normalize(DATASET, mode='none')
H, W, D = data.shape

def false_color(d, bands=(60,30,10)):
    rgb = d[..., list(bands)].astype(np.float32)
    lo = np.percentile(rgb, 2, (0,1), keepdims=True)
    hi = np.percentile(rgb, 98, (0,1), keepdims=True)
    return np.clip((rgb-lo)/(hi-lo+1e-9), 0, 1)

def label_img(g):
    out = np.zeros((*g.shape,3), np.float32)
    for c,hx in CLS_HEX.items(): out[g==c] = to_rgb(hx)
    return out

# scenarios: 0-3 manual, 4-7 random
manual = json.load(open(MANUAL))
rnd    = generate_random_boxes(gt, n=4, min_pixels=BASE['min_pixels'],
                               seeds=tuple(BASE['random_scenario_seeds']))
SCEN   = manual + rnd

fc, lm = false_color(data), label_img(gt)
fig, ax = plt.subplots(1, 2, figsize=(13, 9))
ax[0].imshow(fc); ax[0].set_title('False color (bands 60/30/10)'); ax[0].axis('off')
ax[1].imshow(lm); ax[1].set_title('GT label map');                ax[1].axis('off')
for sidx, col in [(0,'cyan'), (2,'yellow'), (4,'magenta')]:
    for box, ls in [(SCEN[sidx]['train_box'],'--'), (SCEN[sidx]['test_box'],'-')]:
        r0,r1,c0,c1 = box
        for a in ax:
            a.add_patch(plt.Rectangle((c0,r0), c1-c0, r1-r0, fill=False,
                                      edgecolor=col, lw=1.8, ls=ls))
    r0,r1,c0,c1 = SCEN[sidx]['test_box']
    ax[0].text(c0, r0-3, f'scen {sidx}', color=col, fontsize=9, weight='bold')
ax[1].legend(handles=[mpatches.Patch(color=CLS_HEX[c], label=CLS_NAMES[c]) for c in range(10)],
             fontsize=6, loc='center left', bbox_to_anchor=(1.01,0.5))
plt.tight_layout(); plt.show()
print('solid = test box, dashed = train box.  scen0=cyan, scen2=yellow, scen4=magenta')

## Scenario 0 — manual box 0

In [ ]:
# ── Scenario 0 — manual box 0 ─────────────────────────────────────────
# dominant in scene = bitumen
# FOREIGN candidates (classes ABSENT from this scene — pick one as foreign_class):
#   2=meadows, 3=gravel, 5=metal_sheets, 6=bare_soil, 8=bricks
ov = dict(BASE)
ov['scenario_index'] = 0

# ── pick the foreign (out-of-scene) target ──
ov['foreign_class']  = 2     # ← change to any absent class id above

# ── per-scene tuning (uncomment to override BASE) ──
# ov['amplitude']        = 0.15
# ov['target_fraction']  = 0.10
# ov['k']                = 5
# ov['n_budget']         = None
# ov['nmlp_K']           = 7
# ov['active_detectors'] = ['NeighborMLP', 'NeighborMLP-CFAR', 'AMF', 'GMM-Levin']

run_dir_0 = run_and_show(ov)

## Scenario 2 — manual box 2

In [ ]:
# ── Scenario 2 — manual box 2 ─────────────────────────────────────────
# dominant in scene = trees
# FOREIGN candidates (classes ABSENT from this scene — pick one as foreign_class):
#   1=asphalt, 2=meadows, 3=gravel, 5=metal_sheets, 6=bare_soil, 7=bitumen, 8=bricks, 9=shadows
ov = dict(BASE)
ov['scenario_index'] = 2

# ── pick the foreign (out-of-scene) target ──
ov['foreign_class']  = 2     # ← change to any absent class id above

# ── per-scene tuning (uncomment to override BASE) ──
# ov['amplitude']        = 0.15
# ov['target_fraction']  = 0.10
# ov['k']                = 5
# ov['n_budget']         = None
# ov['nmlp_K']           = 7
# ov['active_detectors'] = ['NeighborMLP', 'NeighborMLP-CFAR', 'AMF', 'GMM-Levin']

run_dir_2 = run_and_show(ov)

## Scenario 4 — random box (seed 42)

In [ ]:
# ── Scenario 4 — random box (seed 42) ─────────────────────────────────────────
# dominant in scene = asphalt
# FOREIGN candidates (classes ABSENT from this scene — pick one as foreign_class):
#   2=meadows, 3=gravel, 5=metal_sheets, 6=bare_soil, 7=bitumen, 8=bricks, 9=shadows
ov = dict(BASE)
ov['scenario_index'] = 4

# ── pick the foreign (out-of-scene) target ──
ov['foreign_class']  = 2     # ← change to any absent class id above

# ── per-scene tuning (uncomment to override BASE) ──
# ov['amplitude']        = 0.15
# ov['target_fraction']  = 0.10
# ov['k']                = 5
# ov['n_budget']         = None
# ov['nmlp_K']           = 7
# ov['active_detectors'] = ['NeighborMLP', 'NeighborMLP-CFAR', 'AMF', 'GMM-Levin']

run_dir_4 = run_and_show(ov)

## Re-extract / re-target a saved run (no retraining)

In [ ]:
# Change scoring knobs (foreign_class, amplitude, pfa_target, active_detectors,
# gmm_K, edge_guard ...) on an EXISTING run without retraining the nets.
import subprocess, sys, tempfile, yaml, os, glob

RUN_DIR = run_dir_0   # ← any compare_* folder that has models.pt

SCORING = dict(
    results_dir   = RESULTS,
    dataset       = DATASET,
    manual_boxes_path = MANUAL,
    run_inpatch   = False, run_foreign = True,
    foreign_class = 3,          # ← re-target to gravel, etc.
    amplitude     = 0.15,
    pfa_target    = 0.05,
    active_detectors = ['DSM','NeighborMLP','NeighborMLP-CFAR','AMF','GMM-Levin'],
)
tmp = tempfile.NamedTemporaryFile('w', suffix='.yaml', delete=False)
yaml.dump(SCORING, tmp); tmp.close()
subprocess.run([sys.executable, '-u',
                '/content/proj/experiments/spatial/run_compare.py',
                '--from-models', f'{RUN_DIR}/models.pt', '--config', tmp.name])
os.unlink(tmp.name)
new = sorted(glob.glob(f'{RESULTS}/compare_*'))[-1]
import importlib, run_compare as rc; importlib.reload(rc)
rc.show_plots_from_dir(new, sub='foreign')